# Error Analysis — CCKS2026

Stage 13: 错误挖掘与分析

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

## 1. 加载日志

In [ ]:
LOG_PATH = "../outputs/logs/run_*.jsonl"

logs = []
for f in sorted(Path("../outputs/logs").glob("run_*.jsonl")):
    with open(f, "r", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                logs.append(json.loads(line))
print(f"Loaded {len(logs)} log entries")
df = pd.DataFrame(logs)
df.head()

## 2. 按任务类型统计

In [ ]:
type_counts = df['task_type'].value_counts()
print(type_counts)
type_counts.plot(kind='bar')

## 3. KG 路径错误分析

检查 KG 预测中常见的错误模式。

In [ ]:
kg_logs = df[df['task_type'] == 'knowledge_graph']
print(f"KG samples: {len(kg_logs)}")

# 分析预测长度分布
kg_logs['pred_len'] = kg_logs['prediction'].str.len()
print(kg_logs['pred_len'].describe())

## 4. 检索失败分析

检查 Text QA 中 BM25 检索是否返回了空结果。

In [ ]:
text_logs = df[df['task_type'] == 'multi_hop_qa']
print(f"Text QA samples: {len(text_logs)}")

# 这里需要额外记录 evidence 是否为空
empty_evidence = text_logs['evidence'].apply(lambda x: len(x.strip()) < 10)
print(f"Empty evidence: {empty_evidence.sum()} / {len(text_logs)}")

## 5. 表格解析失败分析

In [ ]:
table_logs = df[df['task_type'] == 'table_qa']
print(f"Table QA samples: {len(table_logs)}")
table_logs.head()

## 6. 幻觉检测

检查预测中是否包含证据中不存在的实体/数字。

In [ ]:
def check_hallucination(row):
    """简单检测：预测中的数字是否在证据中出现"""
    import re
    pred_nums = set(re.findall(r'\d+', row['prediction']))
    evid_nums = set(re.findall(r'\d+', row['evidence']))
    new_nums = pred_nums - evid_nums
    return len(new_nums) > 0, new_nums

df['hallucinated'], df['new_nums'] = zip(*df.apply(check_hallucination, axis=1))
print(f"Hallucinated samples: {df['hallucinated'].sum()} / {len(df)}")
df[df['hallucinated']][['id', 'task_type', 'question', 'prediction']].head(10)

## 7. 统计报告

In [ ]:
print("=" * 60)
print("Error Analysis Summary")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"  - KG: {len(kg_logs)}")
print(f"  - Text QA: {len(text_logs)}")
print(f"  - Table QA: {len(table_logs)}")
print()
print(f"Empty evidence (Text QA): {empty_evidence.sum() if 'empty_evidence' in dir() else 'N/A'}")
print(f"Possible hallucinations: {df['hallucinated'].sum()}")
print()

# 按类型输出 top-5 错误样本
for ttype in ['knowledge_graph', 'multi_hop_qa', 'table_qa']:
    subset = df[df['task_type'] == ttype]
    print(f"\n--- {ttype} 预测长度分布 ---")
    print(subset['pred_len'].describe())